# PS-P1: Quantum-Safe Secure Channel - end-to-end notebook
Qiskit Fall Fest 2026, Phase 1, Post-Quantum Cryptography track. Run from the repository root, top to bottom.

* The channel is **classical software**: ML-KEM / ML-DSA / HKDF / AES-256-GCM on ordinary CPUs. Platforms A/B/C/D are CPU+memory budgets.
* Processors A and B (quantum coupling maps) are built in section 7 for the Phase 1 qualification.
* Sections that need `liboqs-python` or `qiskit` skip themselves cleanly when those are missing.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
try:
    from IPython.display import Image, display
except ImportError:                       # plain-Python fallback
    def display(x): print(x)
    Image = lambda filename, **k: f"[figure] {filename}"

from src import problem, process, metrics, routing, visualization
print(problem.PROBLEM_ID, "-", problem.PROBLEM_TITLE)

PS-P1 - Quantum-Safe Secure Channel


## 1. Parameter sets and available backends
The kit asks for the exact parameter sets. Sets whose library is missing are listed as skipped (install liboqs-python for ML-KEM/ML-DSA).

In [2]:
for k, ps in problem.PARAMETER_SETS.items():
    print(f"{k:15s} {ps.describe()}")
AVAILABLE, SKIPPED = metrics.available_sets(list(problem.PARAMETER_SETS))
print("\navailable:", AVAILABLE)
print("skipped  :", {k: v[:60] + '...' for k, v in SKIPPED.items()})

pqc-L3          ML-KEM-768 + ML-DSA-65 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 3 / SIG 3)
pqc-L1          ML-KEM-512 + ML-DSA-44 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 1 / SIG 2)
pqc-L5          ML-KEM-1024 + ML-DSA-87 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 5 / SIG 5)
classical-p256  ECDH-P-256 + ECDSA-P-256 + HKDF-SHA256 + AES-256-GCM (NIST category n/a (quantum-vulnerable))
hybrid-L3       ECDH-P-256+ML-KEM-768 + ML-DSA-65 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 3 (hybrid) / SIG 3)

available: ['pqc-L3', 'pqc-L1', 'pqc-L5', 'classical-p256', 'hybrid-L3']
skipped  : {}


## 2. Working demo: handshake, message and file transfer over localhost TCP

In [3]:
payloads = problem.locate_payloads()
for k, p in payloads.items():
    print(k, p.size, "bytes", "|", p.source)

for key in AVAILABLE:
    out = process.tcp_demo(key, b"hello from the quantum-safe channel", file=payloads["100KB"].path, chunk_bytes=65536)
    print("\n==", key, "==")
    for name, v in out.items():
        print(f"  {name}: {v}")
display(Image(filename=str(ROOT / "results" / "figures" / "handshake_flow.png")) if (ROOT / "results/figures/handshake_flow.png").exists() else "run section 5 to draw figures")

1KB 1024 bytes | stand-in (seed 42)
100KB 102400 bytes | stand-in (seed 42)
10MB 10485760 bytes | stand-in (seed 42)

== pqc-L3 ==
  parameter_set: ML-KEM-768 + ML-DSA-65 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 3 / SIG 3)
  server_got: b'hello from the quantum-safe channel'
  server_ack: ack: 419c639103c7ea39d50f95f795a0f6437931e09fa118f71dfe87967602821d94
  file_sha256: 00af10bcd533daee6412f303f92c892bf213052c9950ba42b70d2d5f81e05db5
  handshake_sizes: {'size_client_hello': 4584, 'size_kem_pk': 1184, 'size_sig_client': 3309, 'size_server_hello': 4445, 'size_kem_ct': 1088, 'size_sig_server': 3309}
  bytes_on_wire: 111731

== pqc-L1 ==
  parameter_set: ML-KEM-512 + ML-DSA-44 + HKDF-SHA256 + AES-256-GCM (NIST category KEM 1 / SIG 2)
  server_got: b'hello from the quantum-safe channel'
  server_ack: ack: 419c639103c7ea39d50f95f795a0f6437931e09fa118f71dfe87967602821d94
  file_sha256: 00af10bcd533daee6412f303f92c892bf213052c9950ba42b70d2d5f81e05db5
  handshake_sizes: {'size_client_he

## 3. Man-in-the-middle scenarios
An on-path relay swaps KEM keys, corrupts ciphertexts, forges the server hello, flips bits, replays and truncates. Every attack must be detected.

In [4]:
mitm = metrics.run_mitm_table(AVAILABLE)
mitm[["param_set", "scenario", "expected_defence", "attack_detected", "pass"]]

## 4. Benchmark on Platforms A/B/C/D
`QUICK = True` is a smoke run (few repetitions). Set `QUICK = False` for the real numbers (about a minute per parameter set).

In [5]:
QUICK = True
metrics.run_all(quick=QUICK, sets=AVAILABLE)


[1/7] platform specs
[2/7] code size
[3/7] processor graphs (optional quantum module)
[4/7] primitive sizes + ops
[5/7] MITM scenarios
[6/7] chunk-size sweep (Platform D design)
  chunk 1024 ...
  chunk 4096 ...
  chunk 16384 ...
  chunk 65536 ...
  chunk 262144 ...
  chunk 1048576 ...
  chunk None ...
      recommended Platform D record size = 65536; processors/platform_D.json has 65536
[7/7] platform benchmark
  platform A | pqc-L3          | 1KB   ...
  platform A | pqc-L3          | 100KB ...
  platform A | pqc-L3          | 10MB  ...
  platform A | pqc-L1          | 1KB   ...
  platform A | pqc-L1          | 100KB ...
  platform A | pqc-L1          | 10MB  ...
  platform A | pqc-L5          | 1KB   ...
  platform A | pqc-L5          | 100KB ...
  platform A | pqc-L5          | 10MB  ...
  platform A | classical-p256  | 1KB   ...
  platform A | classical-p256  | 100KB ...
  platform A | classical-p256  | 10MB  ...
  platform A | hybrid-L3       | 1KB   ...
  platform A | hybrid-L3 

## 5. Results: laptop (A) vs edge (C), sizes, memory

In [6]:
T = ROOT / "results" / "tables"
head = T / "headline_comparison.csv"
display(pd.read_csv(head).round(3) if head.exists() else "headline needs platforms A and C")
display(pd.read_csv(T / "primitive_sizes_and_ops.csv").round(3))
display(pd.read_csv(T / "predicted_handshake_bytes.csv"))   # predicted from FIPS 203/204 sizes, not measured

  platform             environment       param_set                    kem          sig  handshake_ms_median  handshake_ms_p95  kem_public_key_B  kem_ciphertext_B  signature_B  handshake_bytes_on_wire  ram_heap_peak_kb_10MB  ram_rss_delta_kb_10MB  within_mem_budget_10MB  reps_1KB  handshake_slowdown_vs_ecdh  handshake_bytes_ratio_vs_ecdh
0        A                  laptop  classical-p256             ECDH-P-256  ECDSA-P-256                1.245             2.410                65                65           70                      527              30728.694                30760.0                    True        12                       1.000                          1.000
1        A                  laptop          pqc-L3             ML-KEM-768    ML-DSA-65               16.439            29.224              1184              1088         3309                     9143              30728.687                30944.0                    True        12                      13.200               

In [7]:
made = visualization.make_all()
print(made)
for name in ["headline_laptop_vs_edge.png", "sizes_comparison.png", "memory_by_platform.png"]:
    f = ROOT / "results" / "figures" / name
    if f.exists():
        display(Image(filename=str(f)))

['processor_A_graph.png', 'processor_B_graph.png', 'processor_D_graph.png', 'processor_C_study_graph.png', 'handshake_flow.png', 'processor_D_sabre_mapping.png', 'headline_laptop_vs_edge.png', 'sizes_comparison.png', 'transfer_by_platform.png', 'memory_by_platform.png', 'chunk_sweep.png', 'routing_estimate_ABD.png', 't1_t2_sweep.png']
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


In [8]:
bench = pd.read_csv(T / "pqc_benchmark.csv")
bench[["platform", "backend", "instance_id", "chunk_bytes", "handshake_ms_median", "transfer_ms_median", "tracemalloc_peak_kb",
       "within_mem_budget", "bytes_handshake", "record_overhead_pct", "handshake_correct", "payload_integrity_ok"]].round(3)

## 6. Platform D design
Record size chosen by a rule fixed in advance: the smallest record size reaching 90% of the best in-budget throughput.

In [9]:
display(pd.read_csv(T / "chunk_sweep.csv").round(3))
print(json.dumps(json.loads((T / "chunk_sweep_recommendation.json").read_text()), indent=1))
display(Image(filename=str(ROOT / "results" / "figures" / "chunk_sweep.png")))

     chunk_bytes  n_records  transfer_ms_median  transfer_ms_p95  throughput_MBps  peak_rss_delta_kb  tracemalloc_peak_kb  record_overhead_pct  payload_integrity_ok  cores  reps
0           1024      10241             252.896          268.430           41.463                0.0               23.114                2.051                  True      1     3
1           4096       2561              61.369           68.128          170.865                0.0               41.166                0.513                  True      1     3
2          16384        641              28.700           29.781          365.351                0.0              113.166                0.128                  True      1     3
3          65536        161              18.864           19.704          555.870                0.0              401.083                0.032                  True      1     3
4         262144         41              29.007           29.697          361.493             1028.0          

## 7. Processors A, B, and D (qualification and proposed design)
Processor A (5Q 2016-inspired reference) and B (9Q chorded mesh-loop) meet the qualification specification. Processor D is a team-designed 3x3 planar grid evaluated for comparative multi-path routing analysis.

In [10]:
print(routing.write_processor_files())          # regenerate JSON from the kit spec and validate
for key in ('A', 'B', 'D'):
    proc = routing.load_processor(key)
    print(proc['name'], proc['num_qubits'], 'qubits', proc['coupling_map'])
display(routing.processor_graph_table())
display(routing.routing_estimate_table())
for name in ('processor_A_graph.png', 'processor_B_graph.png', 'processor_D_graph.png', 'processor_D_sabre_mapping.png'):
    display(Image(filename=str(ROOT / 'results' / 'figures' / name)))


{'A': 'OK', 'B': 'OK', 'D': 'OK'}
Processor A (5-qubit 2016-inspired reference) 5 qubits [[0, 1], [1, 2], [2, 3], [1, 4]]
Processor B (9-qubit mesh-loop topology) 9 qubits [[0, 1], [0, 8], [1, 2], [1, 6], [2, 3], [2, 5], [2, 6], [3, 4], [4, 5], [5, 6], [6, 7], [6, 8]]
Processor D (3x3 grid - team design) 9 qubits [[0, 1], [1, 2], [3, 4], [4, 5], [6, 7], [7, 8], [0, 3], [3, 6], [1, 4], [4, 7], [2, 5], [5, 8]]
                                                   processor  required  qubits  edges  max_degree  avg_degree  diameter  avg_shortest_path  avg_swap_lower_bound_all_pairs  max_swap_lower_bound  fully_connected_pairs_pct
0              Processor A (5-qubit 2016-inspired reference)      True       5      4           3       1.600         3              1.800                           0.800                     2                       40.0
1                   Processor B (9-qubit mesh-loop topology)      True       9     12           5       2.667         4              1.944          

## 8. Quantum experiment: Star-GHZ and T1/T2 thermal relaxation sweep on Processors A, B, and D
Executed using Qiskit Aer with the Challenge Kit fixed noise model (depolarizing + readout errors, 4096 shots, seed 42) and extended T1/T2 coherence sweep.

In [11]:
from src.routing import run_probe, run_t1_t2_sweep
probe_df = pd.DataFrame(run_probe())
display(probe_df[['processor', 'cx_count', 'depth', 'swap_count', 'p_success_noisy', 'noisy_standard_error']])
sweep_df = pd.DataFrame(run_t1_t2_sweep())
display(sweep_df)
for name in ('architecture_probe.svg', 'routing_estimate_ABD.png', 't1_t2_sweep.png'):
    display(Image(filename=str(ROOT / 'results' / 'figures' / name)))


  processor  cx_count  depth  swap_count  p_success_noisy  noisy_standard_error
0         A         7     11           1         0.932129              0.003930
1         B         4      8           0         0.940674              0.003691
2         D         4      8           0         0.940674              0.003691
   t1_us  t2_us  gate_time_1q_ns  gate_time_2q_ns  fidelity_A  stderr_A  fidelity_B  stderr_B  fidelity_D  stderr_D
0   15.0   21.0             35.0            250.0      0.8652    0.0053      0.8931    0.0048      0.8936    0.0048
1   30.0   42.0             35.0            250.0      0.9084    0.0045      0.9202    0.0042      0.9199    0.0042
2   50.0   70.0             35.0            250.0      0.9236    0.0042      0.9348    0.0039      0.9351    0.0039
3   80.0  112.0             35.0            250.0      0.9358    0.0038      0.9438    0.0036      0.9438    0.0036
4  120.0  168.0             35.0            250.0      0.9438    0.0036      0.9473    0.0035      0

## 9. Closing checklist (Challenge Kit section 13)
- [ ] one track and one problem statement (PS-P1) - [ ] official payloads used (copy them to `data/raw/`) - [ ] reference documented (ECDH P-256 baseline)
- [ ] implementation runs end to end - [ ] Platforms A/B/C measured, Platform D designed and re-run - [ ] metrics reported: time, memory, code size, bytes exchanged
- [ ] README, report, requirements, results present - [ ] another person can reproduce from the README